==========================================================
AI BASED CAFETERIA FOOD WASTE PREDICTION SYSTEM
==========================================================

==========================
IMPORT LIBRARIES
==========================

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import joblib

==========================================================
STEP 1 : LOAD DATASET
==========================================================

In [2]:
df = pd.read_csv(
    r"C:\Rubina\Msc\Trimester 4\Gen AI\CIA_project\cafeteria_food_waste_dataset.csv"
)
df.rename(
    columns={
        "Temperature_C":"Temperature"
    },
    inplace=True
)
print("\n==============================")
print("DATASET INFORMATION")
print("==============================")
print("Shape :", df.shape)
print("\nColumns:")
print(df.columns)
print("\nFirst 5 rows")
print(df.head())
print("\nMissing Values")
print(df.isnull().sum())


DATASET INFORMATION
Shape : (12000, 29)

Columns:
Index(['Date', 'Meal_Type', 'Day', 'Month', 'Year', 'Students_Registered',
       'Students_Present', 'Faculty_Present', 'Staff_Present', 'Visitors',
       'Hostel_Students', 'Total_Campus_Population', 'Scheduled_Classes',
       'Exam_Period', 'Holiday', 'Semester_Week', 'Weather', 'Temperature',
       'Rainfall_mm', 'Humidity', 'Campus_Event', 'Special_Menu',
       'Menu_Category', 'Meal_Price', 'Previous_Day_Meals_Sold',
       'Previous_Week_Average', 'Meals_Sold', 'Meals_Prepared',
       'Food_Waste_kg'],
      dtype='object')

First 5 rows
         Date  Meal_Type        Day    Month  Year  Students_Registered  \
0  2020-01-01      Lunch  Wednesday  January  2020                 2051   
1  2020-01-02     Dinner   Thursday  January  2020                 2134   
2  2020-01-03      Lunch     Friday  January  2020                 2100   
3  2020-01-04      Lunch   Saturday  January  2020                 1598   
4  2020-01-05  Bre

==========================================================
STEP 2 : DATA PREPROCESSING
==========================================================

Convert Date

In [3]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(
    "Date"
).reset_index(
    drop=True
)

Extract date features

In [4]:
df["Day_Number"] = df["Date"].dt.day
df["Month_Number"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year

7_day_average_demand

In [5]:
df["7_Day_Average"] = (
    df["Previous_Week_Average"]
)
df["7_Day_Average"] = df["7_Day_Average"].fillna(
    df["Previous_Day_Meals_Sold"]
)

Remove date

In [6]:
df.drop(
    "Date",
    axis=1,
    inplace=True
)

==========================
Convert Yes/No values
==========================

In [7]:
for col in df.columns:
    if df[col].dtype == "object":
        unique_values = df[col].dropna().unique()
        if set(unique_values).issubset({"Yes","No"}):
            df[col] = df[col].map(
                {
                    "Yes":1,
                    "No":0
                }
            )

==========================
Encode categories
==========================

In [8]:
categorical_columns = df.select_dtypes(
    include="object"
).columns
encoders={}
for col in categorical_columns:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(
        df[col]
    )
    encoders[col]=encoder
print("\nAfter Encoding")
print(df.head())


After Encoding
   Meal_Type  Day  Month  Year  Students_Registered  Students_Present  \
0          2    6      4  2020                 2051              1441   
1          2    6      4  2020                 2093              1582   
2          2    6      4  2020                 1717              1411   
3          0    6      4  2020                 1381               977   
4          0    6      4  2020                 1457              1125   

   Faculty_Present  Staff_Present  Visitors  Hostel_Students  ...  \
0              197             68        35              552  ...   
1              180             60        46              913  ...   
2              150            115        44              756  ...   
3              150            101        48              384  ...   
4               97             75        35              364  ...   

   Menu_Category  Meal_Price  Previous_Day_Meals_Sold  Previous_Week_Average  \
0              4          57                     2

==========================================================
CHECK REMAINING TEXT DATA
==========================================================

In [9]:
print("\nRemaining Text Columns")
print(
    df.select_dtypes(
        include="object"
    ).columns
)


Remaining Text Columns
Index([], dtype='object')


==========================================================
STEP 3 : CREATE FEATURES AND TARGET
==========================================================

In [10]:
X = df.drop(
    [
        "Meals_Sold",
        "Food_Waste_kg",
        "Meals_Prepared",
    ],
    axis=1
)
y = df["Meals_Sold"]
print("\nFeatures")
print(
    X.columns
)
print("\nFeature Shape")
print(
    X.shape
)
print("\nTarget Shape")
print(
    y.shape
)


Features
Index(['Meal_Type', 'Day', 'Month', 'Year', 'Students_Registered',
       'Students_Present', 'Faculty_Present', 'Staff_Present', 'Visitors',
       'Hostel_Students', 'Total_Campus_Population', 'Scheduled_Classes',
       'Exam_Period', 'Holiday', 'Semester_Week', 'Weather', 'Temperature',
       'Rainfall_mm', 'Humidity', 'Campus_Event', 'Special_Menu',
       'Menu_Category', 'Meal_Price', 'Previous_Day_Meals_Sold',
       'Previous_Week_Average', 'Day_Number', 'Month_Number', '7_Day_Average'],
      dtype='object')

Feature Shape
(12000, 28)

Target Shape
(12000,)


==========================================================
STEP 4 : TRAIN TEST SPLIT
==========================================================

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print("\nTraining Data")
print(
    X_train.shape
)
print("\nTesting Data")
print(
    X_test.shape
)


Training Data
(9600, 28)

Testing Data
(2400, 28)


==========================================================
STEP 5 : RANDOM FOREST MODEL
==========================================================

In [12]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42
)
print("\nTraining Random Forest...")
rf_model.fit(
    X_train,
    y_train
)
rf_prediction = rf_model.predict(
    X_test
)


Training Random Forest...


==========================================================
STEP 6 : XGBOOST MODEL
==========================================================

In [13]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)
print("\nTraining XGBoost...")
xgb_model.fit(
    X_train,
    y_train
)
xgb_prediction = xgb_model.predict(
    X_test
)


Training XGBoost...


==========================================================
STEP 7 : MODEL EVALUATION
==========================================================

In [14]:
def evaluate_model(
        name,
        actual,
        prediction
):
    mae = mean_absolute_error(
        actual,
        prediction
    )
    rmse = np.sqrt(
        mean_squared_error(
            actual,
            prediction
        )
    )
    r2 = r2_score(
        actual,
        prediction
    )
    print("\n====================")
    print(name)
    print("====================")
    print(
        "MAE:",
        round(mae,2)
    )
    print(
        "RMSE:",
        round(rmse,2)
    )
    print(
        "R2 Score:",
        round(r2,4)
    )
evaluate_model(
    "Random Forest",
    y_test,
    rf_prediction
)
evaluate_model(
    "XGBoost",
    y_test,
    xgb_prediction
)


Random Forest
MAE: 58.33
RMSE: 85.75
R2 Score: 0.9808

XGBoost
MAE: 37.64
RMSE: 48.7
R2 Score: 0.9938


==========================================================
STEP 8 : SAVE BEST MODEL
==========================================================

In [15]:
best_model = xgb_model
joblib.dump(
    best_model,
    "cafeteria_xgboost_model.pkl"
)
print("\nModel Saved Successfully")


Model Saved Successfully


==========================================================
STEP 9 : FUTURE DAY PREDICTION
==========================================================

In [16]:
print("\n==============================")
print("FUTURE DAY MEAL PREDICTION")
print("==============================")


FUTURE DAY MEAL PREDICTION


Create future day automatically

In [17]:
future_day = X.mean().to_frame().T

Example future scenario

In [18]:
future_day["Students_Present"] = 1800
future_day["Scheduled_Classes"] = 6

No exam

In [19]:
future_day["Exam_Period"] = 0

No holiday

In [20]:
future_day["Holiday"] = 0

Sunny weather

In [21]:
future_day["Weather"] = encoders["Weather"].transform(
    ["Sunny"]
)[0]
expected_students = int(
    future_day["Students_Present"].values[0]
)
print(
    "Expected Students:",
    expected_students
)
meal_predictions={}
for meal in [
    "Breakfast",
    "Lunch",
    "Dinner"
]:
    meal_code = encoders["Meal_Type"].transform(
        [meal]
    )[0]
    prediction_input = future_day.copy()
    prediction_input["Meal_Type"] = meal_code
    prediction = best_model.predict(
        prediction_input
    )[0]
    meal_predictions[meal]=int(prediction)
print("\nPredicted Meals")
for meal,value in meal_predictions.items():
    print(
        meal,
        "Predicted :",
        value,
        "meals"
    )

Expected Students: 1800

Predicted Meals
Breakfast Predicted : 1046 meals
Lunch Predicted : 1539 meals
Dinner Predicted : 873 meals


==========================================================
STEP 10 : FOOD WASTE CALCULATION
==========================================================

In [22]:
total_demand = sum(
    meal_predictions.values()
)

Traditional cafeteria preparation
15% extra food prepared

In [23]:
without_ai_prepare = int(
    total_demand * 1.15
)

AI optimized preparation
Only 3% extra food prepared

In [24]:
with_ai_prepare = int(
    total_demand * 1.03
)
without_ai_waste = (
    without_ai_prepare - total_demand
)
with_ai_waste = (
    with_ai_prepare - total_demand
)
print("\n==============================")
print("FOOD WASTE ANALYSIS")
print("==============================")
print(
    "Without AI Waste :",
    without_ai_waste,
    "meals"
)
print(
    "With AI Waste :",
    with_ai_waste,
    "meals"
)
reduction = (
    (without_ai_waste - with_ai_waste)
    /
    without_ai_waste
) * 100
print(
    "Waste Reduction :",
    round(reduction,2),
    "%"
)


FOOD WASTE ANALYSIS
Without AI Waste : 518 meals
With AI Waste : 103 meals
Waste Reduction : 80.12 %


==========================================================
STEP 11 : AI CAFETERIA ADVICE
==========================================================

In [25]:
print("\n==============================")
print("AI CAFETERIA ADVICE")
print("==============================")
if with_ai_waste < without_ai_waste:
    print(
"""
AI Recommendation:

1. Prepare meals according to predicted demand.

2. Reduce over-preparation during low attendance days.

3. Increase food preparation during exams and campus events.

4. Consider weather conditions before planning meals.

5. Monitor previous meal demand patterns.

"""
    )
else:
    print(
"""
AI Recommendation:

Increase prediction accuracy by adding more historical cafeteria data.

"""
    )


AI CAFETERIA ADVICE

AI Recommendation:

1. Prepare meals according to predicted demand.

2. Reduce over-preparation during low attendance days.

3. Increase food preparation during exams and campus events.

4. Consider weather conditions before planning meals.

5. Monitor previous meal demand patterns.




==========================================================
STEP 12 : LLAMA 3 NEXT-DAY SPECIAL MENU
==========================================================

In [26]:
import ollama

In [27]:
print("\n==============================")
print("GENERATIVE AI NEXT-DAY MENU")
print("==============================")


GENERATIVE AI NEXT-DAY MENU


In [28]:
menu_prompt = f"""

You are an AI university cafeteria menu planner.

Use the following ML prediction:

Expected Students: {expected_students}

Breakfast demand: {meal_predictions['Breakfast']} meals
Lunch demand: {meal_predictions['Lunch']} meals
Dinner demand: {meal_predictions['Dinner']} meals

Food waste with AI: {with_ai_waste} meals
Waste reduction: {round(reduction, 2)}%

Create ONE practical Indian university cafeteria menu for tomorrow.

The menu must:
- be affordable
- be nutritious
- be attractive to students
- be practical to prepare in bulk
- match the predicted demand
- help reduce food waste

Return ONLY this format:

BREAKFAST: dish
LUNCH: dish
DINNER: dish

Do not provide explanations.
"""

In [29]:
response = ollama.chat(
    model="llama3",
    messages=[
        {
            "role": "user",
            "content": menu_prompt
        }
    ]
)

In [30]:
special_menu = response["message"]["content"]

In [31]:
# Format menu into separate lines
special_menu = special_menu.replace("LUNCH:", "\nLUNCH:")
special_menu = special_menu.replace("DINNER:", "\nDINNER:")

In [32]:
# Save the generated menu
with open(
    r"C:\Users\rubyt\Python\Gen AI\next_day_menu.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(special_menu)

In [33]:
print("\n==============================")
print("NEXT DAY SPECIAL MENU")
print("==============================")
print(special_menu)


NEXT DAY SPECIAL MENU
BREAKFAST: Vegetable Poha Upma

LUNCH: Chana Masala with Brown Rice and Naan

DINNER: Chicken Tikka with Basmati Rice and Mixed Vegetable Curry


In [34]:
print("\nMenu saved successfully.")


Menu saved successfully.
